# Análise de Redes Sociais com Grafos

**Teoria e Aplicação de Grafos, Turma 01**

| Integrante | Matrícula |
|---|---|
| Igor Araújo Rodrigues | 242001517 |


Este notebook analisa a estrutura de uma rede social do Facebook usando métricas de grafos.
Nesta parte são implementadas as etapas:

1. **Coletar dados** — leitura do conjunto *ego-Facebook* (SNAP) e extração aleatória de 404 nós com suas arestas;
2. **Construir grafo** — representação da rede usando NetworkX.

## Instruções

- Os dados devem estar na pasta `facebook/` (mesmo diretório deste notebook), com os arquivos do
  conjunto [ego-Facebook](https://snap.stanford.edu/data/egonets-Facebook.html).
- Dependências: `networkx` e `pandas` (`pip install networkx pandas`).
- Execute as células na ordem.

In [2]:
import os
import random

import networkx as nx
import pandas as pd

## 1. Coletar dados

### 1.1 Leitura do conjunto de dados

O conjunto é formado por 10 redes-ego. Para cada usuário-ego `X` existem, entre outros, os arquivos:

- `X.edges`: arestas entre os amigos de `X` (o próprio ego não aparece);
- `X.feat`: lista dos amigos de `X` (primeira coluna de cada linha), cada um com suas características.

Como o ego é ligado a todos os seus amigos, a rede completa é obtida unindo as arestas de `X.edges`
com as arestas `(X, amigo)` para todo amigo listado em `X.feat`.

In [3]:
PASTA_DADOS = "facebook"

def listar_egos(pasta):
    return sorted(int(nome.split(".")[0]) for nome in os.listdir(pasta) if nome.endswith(".edges"))

def ler_arestas_ego(pasta, ego):
    arestas = []
    with open(os.path.join(pasta, f"{ego}.edges")) as arquivo:
        for linha in arquivo:
            u, v = linha.split()
            arestas.append((int(u), int(v)))
    with open(os.path.join(pasta, f"{ego}.feat")) as arquivo:
        for linha in arquivo:
            amigo = int(linha.split()[0])
            arestas.append((ego, amigo))
    return arestas

egos = listar_egos(PASTA_DADOS)
print("Usuários-ego:", egos)

Usuários-ego: [0, 107, 348, 414, 686, 698, 1684, 1912, 3437, 3980]


In [4]:
rede_completa = nx.Graph()
for ego in egos:
    rede_completa.add_edges_from(ler_arestas_ego(PASTA_DADOS, ego))

print("Nós da rede completa:", rede_completa.number_of_nodes())
print("Arestas da rede completa:", rede_completa.number_of_edges())

Nós da rede completa: 4039
Arestas da rede completa: 88234


A rede completa deve conter **4039 nós** (usuários anonimizados) e 88234 arestas, conforme a descrição do conjunto no SNAP.

### 1.2 Extração aleatória de 404 nós

São sorteados 404 nós da rede completa, sem reposição. Em seguida são extraídas todas as arestas
cujas duas extremidades pertencem aos nós sorteados (subgrafo induzido).

A `SEMENTE` torna o sorteio reproduzível; cada grupo deve usar uma semente própria para obter um
conjunto de dados diferente.

In [5]:
QUANTIDADE_NOS = 404
SEMENTE = 2026

sorteador = random.Random(SEMENTE)
nos_sorteados = sorteador.sample(sorted(rede_completa.nodes()), QUANTIDADE_NOS)

conjunto_sorteados = set(nos_sorteados)
arestas_sorteadas = [
    (u, v) for u, v in rede_completa.edges() if u in conjunto_sorteados and v in conjunto_sorteados
]

print("Nós sorteados:", len(nos_sorteados))
print("Arestas extraídas:", len(arestas_sorteadas))

Nós sorteados: 404
Arestas extraídas: 876


In [6]:
tabela_nos = pd.DataFrame({"no": sorted(nos_sorteados)})
tabela_arestas = pd.DataFrame(arestas_sorteadas, columns=["origem", "destino"])

tabela_arestas.head()

,origem,destino
0,322,104
1,322,185
2,322,21
3,322,345
4,322,200


## 2. Construir grafo

A rede amostrada é representada como um grafo não direcionado do NetworkX (`nx.Graph`), pois a
amizade no Facebook é uma relação recíproca. Todos os 404 nós são adicionados, inclusive os que
ficaram sem arestas após a amostragem (nós isolados).

In [7]:
grafo = nx.Graph()
grafo.add_nodes_from(tabela_nos["no"])
grafo.add_edges_from(tabela_arestas.itertuples(index=False, name=None))

print("Número de nós:", grafo.number_of_nodes())
print("Número de arestas:", grafo.number_of_edges())
print("Densidade:", round(nx.density(grafo), 6))
print("Nós isolados:", nx.number_of_isolates(grafo))
print("Componentes conexas:", nx.number_connected_components(grafo))
print("Tamanho da maior componente:", len(max(nx.connected_components(grafo), key=len)))

Número de nós: 404
Número de arestas: 876
Densidade: 0.010761
Nós isolados: 70
Componentes conexas: 100
Tamanho da maior componente: 75


Resumo dos graus dos nós do grafo construído:

In [8]:
tabela_graus = pd.DataFrame(grafo.degree(), columns=["no", "grau"])
tabela_graus["grau"].describe()

count    404.000000
mean       4.336634
std        5.984309
min        0.000000
25%        1.000000
50%        2.000000
75%        5.250000
max       73.000000
Name: grau, dtype: float64